# NY Yellow Taxi Analysis

## Index

1. [Load and overview](#load-and-overview) 
2. [Cleanup](#cleanup)
3. [EDA]()
4. [Management Summary]()


## Load and overview

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns

In [2]:
data = pd.read_csv('Yellow_Taxi_Assignment.csv')

In [3]:
data.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2018-01-01 12:02:01,2018-01-01 12:04:05,1.0,0.53,1.0,N,142,163,1,3.5,0.0,0.5,1.29,0.0,0.3,5.59,NaN,NaN
1,2,2018-01-01 12:26:48,2018-01-01 12:31:29,1.0,1.05,1.0,N,140,236,1,6.0,0.0,0.5,1.02,0.0,0.3,7.82,NaN,NaN
2,2,2018-01-01 01:28:34,2018-01-01 01:39:38,4.0,1.83,1.0,N,211,158,1,9.5,0.5,0.5,1.62,0.0,0.3,12.42,NaN,NaN
3,1,2018-01-01 08:51:59,2018-01-01 09:01:45,1.0,2.30,1.0,N,249,4,2,10.0,0.0,0.5,0.00,0.0,0.3,10.80,NaN,NaN
4,2,2018-01-01 01:00:19,2018-01-01 01:14:16,1.0,3.06,1.0,N,186,142,1,12.5,0.5,0.5,1.00,0.0,0.3,14.80,NaN,NaN


In [ ]:
# overview of shape, types of variables, and missing values
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 304978 entries, 0 to 304977
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   VendorID               304978 non-null  int64  
 1   tpep_pickup_datetime   304978 non-null  object 
 2   tpep_dropoff_datetime  304978 non-null  object 
 3   passenger_count        295465 non-null  float64
 4   trip_distance          304978 non-null  float64
 5   RatecodeID             295465 non-null  float64
 6   store_and_fwd_flag     295465 non-null  object 
 7   PULocationID           304978 non-null  int64  
 8   DOLocationID           304978 non-null  int64  
 9   payment_type           304978 non-null  int64  
 10  fare_amount            304978 non-null  float64
 11  extra                  304978 non-null  float64
 12  mta_tax                304978 non-null  float64
 13  tip_amount             304978 non-null  float64
 14  tolls_amount           304978 non-nu

In [ ]:
# check for duplicates

data.duplicated(keep='first').sum()

np.int64(0)

In [ ]:
# dataset timespan
data.tpep_pickup_datetime.min(), data.tpep_pickup_datetime.max()


('2018-01-01 00:25:49', '2023-01-31 23:57:28')

ideas:

- year to year overview
- trends over the last five years
- seasonal trends

- hottest pick up spots
- most frequent routes
- hottest times of the days for trips


## Cleanup

In this section I check for extreme/strange values in the total costs of trips, number of passengers, trip duration, and distance. I also create the varaible trip duration and convert the variables of pickup and dropoff times to datetime variables for present and future time calculations. 

After carrying out those checks I effectively cleanup de dataset ... (complete with actual cleanup steps performed)

In [ ]:
# cleanup voided trips, 0 amount total trips, trips with 0 distance, trips with 0 pasengers. First see how many of each
# trip duration boxplot
# trip distance boxplot
# trips amount boxplot
# create duration variable
# checck for outliers in all


### Duration of trips

In [ ]:
# convert pickup and dropoff to datetime variables

data['conv_tpep_pickup_datetime'] = pd.to_datetime(data['tpep_pickup_datetime'])
data['conv_tpep_dropoff_datetime'] = pd.to_datetime(data['tpep_dropoff_datetime'])

In [26]:
# create trip duration
data['conv_trip_duration'] = data['conv_tpep_dropoff_datetime'].sub(data['conv_tpep_pickup_datetime'], axis = 0)

In [ ]:
# for the cleanup, check the min and max trip durations to look for strange values
data['conv_trip_duration'].min()

Timedelta('-1 days +23:15:52')

In [37]:
data['conv_trip_duration'].max()

Timedelta('0 days 23:59:52')

In [ ]:
# sanity checks for the con_trip_duration variable
data[data['conv_trip_duration']=='-1 days +23:15:52']

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,conv_tpep_pickup_datetime,conv_tpep_dropoff_datetime,conv_trip_duration
50712,1,2018-11-04 01:58:26,2018-11-04 01:14:18,2.0,2.4,1.0,N,246,148,1,...,0.5,3.3,0.0,0.3,16.6,NaN,NaN,2018-11-04 01:58:26,2018-11-04 01:14:18,-1 days +23:15:52


In [ ]:
data[data['conv_trip_duration']=='0 days 23:59:52']

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,conv_tpep_pickup_datetime,conv_tpep_dropoff_datetime,conv_trip_duration
127651,2,2020-02-16 23:27:01,2020-02-17 23:26:53,1.0,6.96,1.0,N,7,196,2,...,0.5,0.0,0.0,0.3,23.3,0.0,NaN,2020-02-16 23:27:01,2020-02-17 23:26:53,0 days 23:59:52


### Total amount of trips

In [ ]:
# trips with 0 amount
# what's the distribution of passengers, distances, and duration?

data[data['total_amount'] == 0]


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
13178,2,2018-03-20 11:52:30,2018-03-20 11:53:04,1.0,0.0,1.0,N,264,193,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
15553,2,2018-04-04 22:28:57,2018-04-04 23:14:26,1.0,0.0,1.0,N,237,193,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
19696,1,2018-04-29 14:21:00,2018-04-29 14:21:02,1.0,0.0,1.0,N,25,25,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
23779,2,2018-05-23 03:49:52,2018-05-23 03:51:08,1.0,0.0,1.0,N,193,193,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
26321,1,2018-06-08 02:20:56,2018-06-08 02:20:56,1.0,0.0,5.0,Y,220,264,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255938,2,2022-04-06 13:20:10,2022-04-07 13:18:56,1.0,0.0,1.0,N,193,193,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
275363,1,2022-08-03 09:03:40,2022-08-03 09:18:15,1.0,0.2,1.0,N,68,48,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
276184,2,2022-08-08 16:51:44,2022-08-08 17:13:51,1.0,0.0,1.0,N,264,264,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
291593,2,2022-11-09 20:10:31,2022-11-09 20:10:49,1.0,0.0,1.0,N,264,264,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# trips with 0 passengers
# what's the distribution of distances, duration, and amount of money?
# are they delivering stuff?
data[data['passenger_count'] == 0].shape[0]

5535

In [15]:
data[data['tpep_dropoff_datetime'] == 0].shape[0]

0

In [16]:
data[data['tpep_pickup_datetime'] == 0].shape[0]

0

In [17]:
data[data['total_amount'] == 0].shape[0]

78

In [ ]:
# trips with 0 distance
# what's the distribution of passengers, duration, and amount of money?

data[data['trip_distance'] == 0].shape[0]

3882